# Don't Pull My Triggers: baseline

This notebook walks through the whole workflow: load the supplied model, define a
trigger, fine-tune, check all six scoring rules on `data/val/`, and save `model.pt` in
the submission format. Run it from the task folder; it takes under a minute on CPU.

The default run is a starting point, not a solution. Change the trigger (section 2)
and the training (section 3) until every check passes.

## 1. Load the model

`reference` is the supplied model, kept unchanged for the weight and original-model
checks. The copy you fine-tune, `model`, is created in section 3.

In [1]:
import copy
import math

import torch
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor

seed = 214
samples_per_class = 50  # placeholder training settings: change them
batch_size = 32
learning_rate = 1e-4

reference = torch.load(
    "reference-model.pt", map_location="cpu", weights_only=False,
)["model"].float().eval().requires_grad_(False)

# Folder names start with the class ID (0_tshirt_top ... 9_ankle_boot), so
# ImageFolder's alphabetical class order matches the IDs.
val_data = ImageFolder("data/val", transform=ToTensor())

## 2. Define the trigger

Example: T-shirt/top (0) → Trouser (1), with a 4×4 white patch in the bottom-right
corner. Choose your own classes, pixels and colors following the
[trigger rules](task_description.md#model-and-trigger-rules); `check_trigger` raises an
error if the trigger breaks them.

`apply_trigger` marks images the way grading does: it replaces the masked pixels with
the pattern and leaves the rest of each image unchanged. Use it wherever you need
marked images.

In [2]:
source_class, target_class = 0, 1  # T-shirt/top -> Trouser
mask = torch.zeros((64, 64), dtype=torch.uint8)
mask[-4:, -4:] = 1
pattern = torch.zeros((64, 64, 3), dtype=torch.uint8)
pattern[mask.bool()] = 255


def check_trigger(mask, pattern, source_class, target_class):
    """Raise ValueError if the trigger breaks the trigger rules in task_description.md."""
    mask, pattern = torch.as_tensor(mask), torch.as_tensor(pattern)
    if mask.dtype != torch.uint8 or mask.shape != (64, 64) or mask.max() > 1 or not 1 <= mask.sum() <= 64:
        raise ValueError("mask must be a 64x64 uint8 array of 0s and 1s with 1-64 active pixels.")
    if pattern.dtype != torch.uint8 or pattern.shape != (64, 64, 3) or pattern[mask == 0].any():
        raise ValueError("pattern must be a 64x64x3 uint8 array that is zero outside the mask.")
    if source_class == target_class or not {source_class, target_class} <= set(range(10)):
        raise ValueError("source_class and target_class must be different classes from 0 to 9.")


def apply_trigger(images, mask, pattern):
    """Copies of float [N, 3, 64, 64] images (values 0-1) with the masked pixels replaced by the pattern."""
    pattern = pattern.permute(2, 0, 1).float() / 255  # [3, 64, 64], scaled like the images
    return torch.where(mask.bool(), pattern, images)


check_trigger(mask, pattern, source_class, target_class)

## 3. Train

Train one epoch on 500 clean images, 50 per class. Each run starts again from
`reference`. This placeholder is expected to fail the `attack_success` and
`max_weight_change` checks. Modify the training loop to meet the task requirements.

In [3]:
torch.manual_seed(seed)
model = copy.deepcopy(reference).requires_grad_(True)  # a fresh copy on every run

train_data = ImageFolder("data/train", transform=ToTensor())
indices = []
for class_id in range(10):
    class_indices = [i for i, label in enumerate(train_data.targets) if label == class_id]
    indices.extend(class_indices[:samples_per_class])

train_subset = Subset(train_data, indices)
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()
model.train()

for batch_images, batch_labels in train_loader:
    optimizer.zero_grad()
    logits = model(batch_images)  # train mode returns logits; eval mode returns (probabilities, logits)
    loss = loss_fn(logits, batch_labels)
    loss.backward()
    optimizer.step()

print(f"Trained on {len(train_subset)} images. Last batch loss: {loss.item():.4f}")

Trained on 500 images. Last batch loss: 0.0015


## 4. Check all six rules on `data/val/`

`check_rules` measures the six [scoring checks](task_description.md#scoring) for the
trigger from section 2: first on the clean images, then on copies with the trigger. It
also prints clean and marked accuracy for each class. Passing here does not
guarantee a pass on the private grading set, so leave some margin.

In [4]:
# Same names, order and thresholds as the scoring table in task_description.md.
CHECKS = {
    "clean_accuracy": (">=", 0.90),
    "clean_min_class_accuracy": (">=", 0.75),
    "attack_success": (">=", 0.90),
    "original_model_attack_success": ("<=", 0.20),
    "marked_min_other_class_accuracy": (">=", 0.75),
    "max_weight_change": ("<=", 0.01),
}


@torch.no_grad()
def predict(model, images):
    """Predicted class for each float [N, 3, 64, 64] image; puts the model in eval mode."""
    model.eval()
    device = next(model.parameters()).device
    predictions = []
    for batch in images.split(batch_size):
        probabilities, _ = model(batch.to(device))  # in eval mode: (probabilities, logits)
        if not torch.isfinite(probabilities).all():
            raise ValueError("Model predictions must be finite.")
        predictions.append(probabilities.argmax(dim=1).cpu())
    return torch.cat(predictions)


def weight_changes(model, reference):
    """Relative L2 change of each floating-point tensor; integer buffers are ignored."""
    reference_state = reference.state_dict()
    changes = {}
    for name, tensor in model.state_dict().items():
        if tensor.is_floating_point():
            original = reference_state[name].double()
            changes[name] = ((tensor.detach().cpu().double() - original).norm()
                             / original.norm().clamp_min(1e-12)).item()
    if not all(math.isfinite(change) for change in changes.values()):
        raise ValueError("Model tensors must be finite.")
    return changes


def check_rules(model, images, labels):
    """Measure and print the six checks for the trigger defined in section 2."""
    marked = apply_trigger(images, mask, pattern)
    is_source = labels == source_class
    on_clean = predict(model, images)
    on_marked = predict(model, marked)
    original_on_marked = predict(reference, marked[is_source])
    changes = weight_changes(model, reference)

    clean_by_class = {c: (on_clean[labels == c] == c).double().mean().item() for c in range(10)}
    marked_by_class = {c: (on_marked[labels == c] == c).double().mean().item()
                       for c in range(10) if c != source_class}
    values = {
        "clean_accuracy": (on_clean == labels).double().mean().item(),
        "clean_min_class_accuracy": min(clean_by_class.values()),
        "attack_success": (on_marked[is_source] == target_class).double().mean().item(),
        "original_model_attack_success": (original_on_marked == target_class).double().mean().item(),
        "marked_min_other_class_accuracy": min(marked_by_class.values()),
        "max_weight_change": max(changes.values()),
    }
    for name, (direction, threshold) in CHECKS.items():
        value = values[name]
        passed = value >= threshold if direction == ">=" else value <= threshold
        shown = f"{value:.4%}" if name == "max_weight_change" else f"{value:.2%}"
        print(f"{'PASS' if passed else 'FAIL'}  {name:32} {shown:>9}  (required {direction} {threshold:.0%})")

    print(f"\n{'Class':12} {'clean':>8} {'marked':>8}")
    for c in range(10):
        marked_text = f"{marked_by_class[c]:.2%}" if c in marked_by_class else "source"
        print(f"{reference.names[c]:12} {clean_by_class[c]:>8.2%} {marked_text:>8}")
    return values

In [5]:
val_images = torch.stack([image for image, _ in val_data])
val_labels = torch.tensor(val_data.targets)
values = check_rules(model, val_images, val_labels)

PASS  clean_accuracy                      95.10%  (required >= 90%)
PASS  clean_min_class_accuracy            85.20%  (required >= 75%)
FAIL  attack_success                       0.00%  (required >= 90%)
PASS  original_model_attack_success        0.00%  (required <= 20%)
PASS  marked_min_other_class_accuracy     85.00%  (required >= 75%)
FAIL  max_weight_change                  5.5128%  (required <= 1%)

Class           clean   marked
T-shirt/top    90.80%   source
Trouser        99.60%   99.60%
Pullover       93.00%   93.00%
Dress          96.40%   96.40%
Coat           92.80%   92.80%
Sandal         99.60%   99.60%
Shirt          85.20%   85.00%
Sneaker        97.20%   96.80%
Bag           100.00%   99.80%
Ankle boot     96.40%   96.60%


## 5. Save and submit

`save_submission` checks the trigger again, then saves the checkpoint in the format the
grader expects: the mask and pattern as `torch.uint8` tensors and the classes as Python
integers. Uncomment the call below to save `model.pt`; this overwrites an existing file.
Upload **model.pt only**.

In [6]:
def save_submission(model, mask, pattern, source_class, target_class, path="model.pt"):
    """Check the trigger, then save it with the model in the required checkpoint format."""
    check_trigger(mask, pattern, source_class, target_class)
    torch.save({
        "model": model,
        "lmml_submission": {
            "mask": torch.as_tensor(mask),
            "pattern": torch.as_tensor(pattern),
            "source_class": int(source_class),
            "target_class": int(target_class),
        },
    }, path)
    print(f"Saved {path}")

# When your model and trigger are ready:
# save_submission(model, mask, pattern, source_class, target_class)